In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def load_data():
    xls = pd.ExcelFile("/kaggle/input/competitions/cohort-x-task-3/Task_3.xlsx")
    test = pd.read_excel(xls, sheet_name='Test')
    icd = pd.read_excel("/kaggle/input/competitions/cohort-x-task-3/mimic-iv_icd-10_dict.xlsx")
    return test, icd

def preprocess(test, icd):
    test["Condition_lower"] = test["Condition"].astype(str).str.lower()
    icd["long_title"] = icd["long_title"].astype(str).str.lower()
    return test, icd

def embed_texts(texts, model):
    return model.encode(texts, batch_size=64, convert_to_numpy=True, show_progress_bar=True)

def assign_dynamic_labels(sims, icd_codes):
    keep, assoc, diff = [], [], []
    mean_sim = sims.mean()
    std_sim = sims.std()
    for code, sim in zip(icd_codes, sims):
        if sim >= mean_sim + 0.5 * std_sim:
            keep.append(code)
        elif sim >= mean_sim:
            assoc.append(code)
        else:
            diff.append(code)
    if not keep:
        keep.append(icd_codes[np.argmax(sims)])
        if keep[0] in diff:
            diff.remove(keep[0])
    fmt = lambda x: "; ".join(x) if x else "Not Applicable"
    return fmt(keep), fmt(assoc), fmt(diff)

def predict(test, icd, cond_emb, icd_emb, top_k=15):
    results = []
    for idx, row in test.iterrows():
        cond_vector = cond_emb[idx].reshape(1, -1)
        sims = cosine_similarity(cond_vector, icd_emb)[0]
        top_idx = sims.argsort()[-top_k:][::-1]
        top_codes = icd.iloc[top_idx]["icd_code"].tolist()
        top_sims = sims[top_idx]
        keep, assoc, diff = assign_dynamic_labels(top_sims, top_codes)
        results.append({
            "Condition": row["Condition"],
            "KEEP": keep,
            "ASSOCIATION": assoc,
            "DIFF": diff
        })
    return pd.DataFrame(results)

def main():
    test, icd = load_data()
    test, icd = preprocess(test, icd)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    cond_emb = embed_texts(test["Condition_lower"].tolist(), model)
    icd_emb = embed_texts(icd["long_title"].tolist(), model)
    submission = predict(test, icd, cond_emb, icd_emb, top_k=15)
    submission.to_csv("submission.csv", index=False)
    return submission

submission = main()
submission.head()